# NB13 — ENIGMA Isolate Site-Level Validation (Exploratory)

**Purpose:** Exploratory site-level validation of the metal-gene/niche hypothesis using  
ENIGMA isolate genomes from the Oak Ridge FRC groundwater site.  
Tests whether isolate genomes from high-metal wells carry more metal-resistance genes  
per Mb than those from low-metal wells.

**Differs from NB11:** NB11 used metagenome-assembled genomes (MAGs; `strain_id IS NULL`).  
This notebook uses **isolate genomes** (`strain_id IS NOT NULL`): pure-culture isolates  
obtained from specific FRC groundwater wells.

**Hypothesis (pre-specified):** ρ > 0 — isolates from higher-metal wells carry higher  
140-KO density per Mb (positive association, opposite sign to niche-breadth H1 because  
here we're measuring gene content, not niche breadth).

**Label:** EXPLORATORY. Not confirmatory. Results should be interpreted cautiously.  
Small n expected. Null/opposite results reported honestly.

**Output:** `data/enigma_isolate_site_validation.csv`

## Block 0 — Imports and Spark

In [1]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

PROJECT = Path('/home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology')
DATA = PROJECT / 'data'

# Pre-specified primary metals for FDR correction
PRIMARY_METALS_FDR = ['Cu', 'Ni', 'Zn']

# Groundwater metal columns in ddt_brick0000007 (mg/L)
METAL_COLS = {
    'Cu': 'concentration_molecule_from_list_copper_atom_milligram_per_liter',
    'Ni': 'concentration_molecule_from_list_nickel_atom_milligram_per_liter',
    'Zn': 'concentration_molecule_from_list_zinc_atom_milligram_per_liter',
    'As': 'concentration_molecule_from_list_arsane_milligram_per_liter',
    'Mn': 'concentration_molecule_from_list_manganese_atom_milligram_per_liter',
    'Cr': 'concentration_molecule_from_list_chromium_atom_milligram_per_liter',
    'Co': 'concentration_molecule_from_list_cobalt_atom_milligram_per_liter',
}

print('Imports OK')

Imports OK


In [3]:
# Spark session: JupyterHub context vs standalone
spark = get_spark_session()

print(f'Spark version: {spark.version}')

Spark version: 4.0.1


## Block 1 — Schema Discovery

In [4]:
# Confirm browser_genome schema and verify isolate/MAG distinction
def _describe(ns, tbl):
    df = spark.sql(f'DESCRIBE TABLE {ns}.{tbl}').toPandas()
    cols = [r for r in df['col_name'].tolist() if not str(r).startswith('#') and str(r).strip()]
    print(f'--- {ns}.{tbl} ({len(cols)} cols) ---')
    print(', '.join(cols))
    return cols

_describe('enigma_genome_depot_enigma', 'browser_genome')
_describe('enigma_genome_depot_enigma', 'browser_sample')

--- enigma_genome_depot_enigma.browser_genome (14 cols) ---
id, name, description, contigs, size, genes, json_url, pub_date, external_url, external_id, gbk_filepath, sample_id, strain_id, taxon_id


--- enigma_genome_depot_enigma.browser_sample (4 cols) ---
id, sample_id, full_name, description


['id', 'sample_id', 'full_name', 'description']

In [5]:
# Verify MAG vs isolate counts
# MAGs: sample_id IS NOT NULL, strain_id IS NULL
# Isolates: sample_id IS NOT NULL, strain_id IS NOT NULL
counts = spark.sql('''
    SELECT
        COUNT(*) AS total_genomes,
        SUM(CASE WHEN sample_id IS NOT NULL AND strain_id IS NULL THEN 1 ELSE 0 END) AS n_mags,
        SUM(CASE WHEN sample_id IS NOT NULL AND strain_id IS NOT NULL THEN 1 ELSE 0 END) AS n_isolates_with_sample,
        SUM(CASE WHEN sample_id IS NULL AND strain_id IS NOT NULL THEN 1 ELSE 0 END) AS n_isolates_no_sample,
        SUM(CASE WHEN sample_id IS NULL AND strain_id IS NULL THEN 1 ELSE 0 END) AS n_neither
    FROM enigma_genome_depot_enigma.browser_genome
''').toPandas()
print('Genome type breakdown:')
print(counts.to_string(index=False))
print()
print('Filter for isolates: sample_id IS NOT NULL AND strain_id IS NOT NULL')

Genome type breakdown:
 total_genomes  n_mags  n_isolates_with_sample  n_isolates_no_sample  n_neither
          3110     185                       0                  2925          0

Filter for isolates: sample_id IS NOT NULL AND strain_id IS NOT NULL


In [6]:
# Preview a few isolate genomes
preview = spark.sql('''
    SELECT g.id, g.name, g.size, g.genes, g.sample_id, g.strain_id, s.sample_id AS well_name
    FROM enigma_genome_depot_enigma.browser_genome g
    JOIN enigma_genome_depot_enigma.browser_sample s ON s.id = g.sample_id
    WHERE g.sample_id IS NOT NULL AND g.strain_id IS NOT NULL
    LIMIT 10
''').toPandas()
print('Sample isolate rows:')
print(preview.to_string(index=False))

Sample isolate rows:
Empty DataFrame
Columns: [id, name, size, genes, sample_id, strain_id, well_name]
Index: []


## Block 2 — Load Primary KO Set (140 KOs)

In [ ]:
gl = pd.read_csv(DATA / 'curated_mrg_ko_ids_v2.csv')
# Use exact membership — startswith('Tier 2') also matches 'Tier 2-Fitness' (256 instead of 140)
primary_kos = set(
    gl.loc[gl['evidence_tier'].isin(['Tier 1', 'Tier 2']), 'KO'].str.strip()
)
print(f'Primary KO set (Tier 1+2): {len(primary_kos)} KOs (expected: 140)')
print('Sample:', sorted(list(primary_kos))[:5])

In [ ]:
# Check browser_kegg_ortholog format
r_ko_fmt = spark.sql('''
    SELECT kegg_id, description
    FROM enigma_genome_depot_enigma.browser_kegg_ortholog
    LIMIT 5
''').toPandas()
print('browser_kegg_ortholog sample rows:')
print(r_ko_fmt.to_string(index=False))

# Adjust format
sample_kegg_id = r_ko_fmt['kegg_id'].iloc[0]
if sample_kegg_id.startswith('ko:'):
    ko_sql_vals = ','.join(f"'ko:{k}'" for k in primary_kos)
else:
    ko_sql_vals = ','.join(f"'{k}'" for k in primary_kos)
print(f'\nKegg ID format: {repr(sample_kegg_id)}')
print(f'SQL values sample: {ko_sql_vals[:80]}...')

## Block 3 — Query Isolate KO Densities

Join chain: `browser_genome → browser_sample` (well) + `browser_genome → browser_gene → browser_protein_kegg_orthologs → browser_kegg_ortholog`  
Filter: `sample_id IS NOT NULL AND strain_id IS NOT NULL` (isolates only)

In [ ]:
isolate_ko_query = f'''
    SELECT
        g.id           AS genome_id,
        g.name         AS genome_name,
        g.size         AS genome_size_bp,
        g.genes        AS n_genes,
        s.sample_id    AS well_id,
        COUNT(DISTINCT ko.kegg_id) AS n_primary_ko
    FROM enigma_genome_depot_enigma.browser_genome g
    JOIN enigma_genome_depot_enigma.browser_sample s
        ON s.id = g.sample_id
    JOIN enigma_genome_depot_enigma.browser_gene gene
        ON gene.genome_id = g.id
    JOIN enigma_genome_depot_enigma.browser_protein_kegg_orthologs pko
        ON pko.protein_id = gene.protein_id
    JOIN enigma_genome_depot_enigma.browser_kegg_ortholog ko
        ON ko.id = pko.kegg_ortholog_id
    WHERE
        g.sample_id IS NOT NULL
        AND g.strain_id IS NOT NULL
        AND ko.kegg_id IN ({ko_sql_vals})
    GROUP BY g.id, g.name, g.size, g.genes, s.sample_id
'''

iso_ko_df = spark.sql(isolate_ko_query).toPandas()
print(f'Isolates with ≥1 primary KO: {len(iso_ko_df)}')
print(iso_ko_df.head())

In [ ]:
# Total isolate count (including those with 0 primary KOs)
all_isolates = spark.sql('''
    SELECT g.id AS genome_id, g.name AS genome_name, g.size AS genome_size_bp, s.sample_id AS well_id
    FROM enigma_genome_depot_enigma.browser_genome g
    JOIN enigma_genome_depot_enigma.browser_sample s ON s.id = g.sample_id
    WHERE g.sample_id IS NOT NULL AND g.strain_id IS NOT NULL
''').toPandas()

print(f'Total isolates: {len(all_isolates)}')
print(f'Isolates with ≥1 primary KO: {len(iso_ko_df)} ({100*len(iso_ko_df)/max(len(all_isolates),1):.1f}%)')
print(f'Wells represented: {all_isolates["well_id"].nunique()}')

# Left join: include isolates with 0 KOs
iso_density = all_isolates.merge(
    iso_ko_df[['genome_id', 'n_primary_ko']], on='genome_id', how='left'
)
iso_density['n_primary_ko'] = iso_density['n_primary_ko'].fillna(0).astype(int)
iso_density['genome_size_mb'] = iso_density['genome_size_bp'] / 1e6
iso_density['ko_per_mb'] = iso_density.apply(
    lambda r: r['n_primary_ko'] / r['genome_size_mb'] if r['genome_size_mb'] > 0 else np.nan,
    axis=1
)

print(f'\nKO density summary:')
print(iso_density['ko_per_mb'].describe().round(3))
print(f'\nPer-well isolate counts:')
print(iso_density.groupby('well_id')['genome_id'].count().sort_values(ascending=False).head(10))

## Block 4 — Fetch Groundwater Geochemistry

In [ ]:
# Fetch groundwater metals from enigma_coral.ddt_brick0000007
# Well ID in 'sdt_sample_name'; match to isolate well_ids via startswith (longest first)
metal_select = ', '.join(f'{v} AS {k}' for k, v in METAL_COLS.items())
geo_raw = spark.sql(f'''
    SELECT sdt_sample_name, {metal_select}
    FROM enigma_coral.ddt_brick0000007
    WHERE sdt_sample_name IS NOT NULL
''').toPandas()

print(f'Geochemistry rows: {len(geo_raw)}')
print(f'Unique sample names: {geo_raw["sdt_sample_name"].nunique()}')
print('\nSample names preview:', geo_raw['sdt_sample_name'].unique()[:10])

In [ ]:
# Match geochemistry sample names to isolate well IDs (longest match first)
all_well_ids = sorted(iso_density['well_id'].unique().tolist(), key=len, reverse=True)
print(f'Isolate well IDs ({len(all_well_ids)}):', all_well_ids)

def _match_well(sample_name, well_ids):
    for w in well_ids:
        if str(sample_name).startswith(w):
            return w
    return None

geo_raw['well_id'] = geo_raw['sdt_sample_name'].apply(
    lambda x: _match_well(str(x), all_well_ids)
)

matched = geo_raw['well_id'].notna().sum()
print(f'\nGeochemistry rows matched to isolate wells: {matched}/{len(geo_raw)} ({100*matched/len(geo_raw):.1f}%)')
print('Matched wells:', sorted(geo_raw.dropna(subset=['well_id'])['well_id'].unique().tolist()))

In [ ]:
# Aggregate geochemistry per well: median across time points
geo_well = (
    geo_raw
    .dropna(subset=['well_id'])
    .groupby('well_id')[list(METAL_COLS.keys())]
    .median()
    .reset_index()
)

geo_counts = (
    geo_raw
    .dropna(subset=['well_id'])
    .groupby('well_id')[list(METAL_COLS.keys())]
    .count()
    .reset_index()
    .rename(columns={m: f'{m}_n' for m in METAL_COLS.keys()})
)
geo_well = geo_well.merge(geo_counts, on='well_id')

print(f'Wells with geochemistry data: {len(geo_well)}')
print(geo_well[['well_id'] + list(METAL_COLS.keys())].to_string(index=False))

## Block 5 — Join Isolate Density to Geochemistry

In [ ]:
# Isolate-level join
iso_geo = iso_density.merge(geo_well, on='well_id', how='inner')

print(f'Isolates with geochemistry: {len(iso_geo)}')
print(f'Wells in joined dataset: {iso_geo["well_id"].nunique()}')
print(f'Isolates dropped (no geo match): {len(iso_density) - len(iso_geo)}')
print()

well_summary = (
    iso_geo.groupby('well_id')
    .agg(n_isolates=('genome_id', 'count'),
         mean_ko_per_mb=('ko_per_mb', 'mean'),
         median_ko_per_mb=('ko_per_mb', 'median'))
    .reset_index()
)
print('Per-well isolate summary:')
print(well_summary.to_string(index=False))

In [ ]:
# Feasibility check — report if n is too small for reliable inference
n_iso = len(iso_geo)
n_wells = iso_geo['well_id'].nunique()
n_wells_ge2 = (iso_geo.groupby('well_id')['genome_id'].count() >= 2).sum()

print(f'Feasibility summary:')
print(f'  Isolates in geo-matched wells: {n_iso}')
print(f'  Unique wells: {n_wells}')
print(f'  Wells with ≥2 isolates (usable for well-level analysis): {n_wells_ge2}')
print()
if n_iso < 5:
    print('WARNING: n_isolates < 5. Spearman analysis will be skipped or marked as infeasible.')
if n_wells < 5:
    print(f'WARNING: only {n_wells} wells with geochemistry data. Well-level analysis is underpowered.')

## Block 6 — Spearman Correlations

**Pre-specified direction:** ρ > 0 (higher metal → higher KO density).  
BH-FDR across 3 primary metals (Cu, Ni, Zn) at both levels.

In [ ]:
def spearman_row(data, metal, density_col='ko_per_mb', level='isolate'):
    sub = data[[density_col, metal]].dropna()
    n = len(sub)
    if n < 5:
        return {'metal': metal, 'level': level, 'n': n, 'rho': np.nan,
                'p_raw': np.nan, 'note': 'n<5 — skip'}
    rho, p2 = stats.spearmanr(sub[density_col], sub[metal])
    return {'metal': metal, 'level': level, 'n': n,
            'rho': round(rho, 4), 'p_raw': round(p2, 4), 'note': ''}

# Isolate-level Spearman
iso_results = []
for metal in METAL_COLS:
    row = spearman_row(iso_geo, metal, density_col='ko_per_mb', level='isolate')
    iso_results.append(row)

iso_corr = pd.DataFrame(iso_results)

# BH FDR on primary metals only (Cu, Ni, Zn)
primary_mask = iso_corr['metal'].isin(PRIMARY_METALS_FDR) & iso_corr['p_raw'].notna()
if primary_mask.sum() > 0:
    _, q_bh, _, _ = multipletests(iso_corr.loc[primary_mask, 'p_raw'].values, method='fdr_bh')
    iso_corr.loc[primary_mask, 'p_fdr'] = q_bh.round(4)

print('Isolate-level Spearman correlations (density vs well metal):')
print(iso_corr.to_string(index=False))

In [ ]:
# Well-level Spearman (wells with ≥2 isolates)
# Aggregate: median KO density per well
well_density = (
    iso_geo.groupby('well_id')
    .agg(n_isolates=('genome_id', 'count'),
         median_ko_per_mb=('ko_per_mb', 'median'))
    .reset_index()
)
well_geo = well_density[well_density['n_isolates'] >= 2].merge(geo_well, on='well_id', how='inner')

print(f'Wells with ≥2 isolates and geo data: {len(well_geo)}')
if len(well_geo) > 0:
    print(well_geo[['well_id', 'n_isolates', 'median_ko_per_mb'] + list(METAL_COLS.keys())].to_string(index=False))

well_results = []
for metal in METAL_COLS:
    row = spearman_row(well_geo, metal, density_col='median_ko_per_mb', level='well')
    well_results.append(row)

well_corr = pd.DataFrame(well_results)

primary_mask_w = well_corr['metal'].isin(PRIMARY_METALS_FDR) & well_corr['p_raw'].notna()
if primary_mask_w.sum() > 0:
    _, q_bh_w, _, _ = multipletests(well_corr.loc[primary_mask_w, 'p_raw'].values, method='fdr_bh')
    well_corr.loc[primary_mask_w, 'p_fdr'] = q_bh_w.round(4)

print('\nWell-level Spearman correlations (median KO density per well):')
print(well_corr.to_string(index=False))

## Block 7 — Save Results

In [ ]:
# Combine isolate and well results
combined = pd.concat([iso_corr, well_corr], ignore_index=True)
combined = combined.rename(columns={'rho': 'spearman_rho'})

# Pre-specified output columns
out_cols = ['metal', 'level', 'n', 'spearman_rho', 'p_raw', 'p_fdr']
out_df = combined[[c for c in out_cols if c in combined.columns]]

out_path = DATA / 'enigma_isolate_site_validation.csv'
out_df.to_csv(out_path, index=False)
print(f'Saved to {out_path}')
print(out_df.to_string(index=False))

## Block 8 — Interpretation and INTERPRETATION_TABLE.md Update

In [ ]:
# Print plain-language interpretation
print('EXPLORATORY INTERPRETATION — ENIGMA Isolate Site-Level Validation')
print('=' * 70)
print(f'n_isolates (matched to geo): {n_iso}')
print(f'n_wells_total: {n_wells}')
print(f'n_wells_ge2_isolates: {n_wells_ge2}')
print()
print('Pre-specified direction: ρ > 0 (higher metal → higher KO density)')
print()

# Primary metals summary
for metal in PRIMARY_METALS_FDR:
    iso_row = iso_corr[iso_corr['metal'] == metal].iloc[0]
    well_row = well_corr[well_corr['metal'] == metal].iloc[0]
    iso_dir = 'consistent' if pd.notna(iso_row['rho']) and iso_row['rho'] > 0 else (
        'inconsistent' if pd.notna(iso_row['rho']) else 'insufficient n')
    print(f'{metal}:')
    print(f'  Isolate: ρ={iso_row["rho"]} p={iso_row["p_raw"]} n={iso_row["n"]} [{iso_dir}]')
    print(f'  Well:    ρ={well_row["rho"]} p={well_row["p_raw"]} n={well_row["n"]}')

In [ ]:
# Update INTERPRETATION_TABLE.md
interp_path = PROJECT / 'INTERPRETATION_TABLE.md'
with open(interp_path, 'r') as f:
    content = f.read()

# Build result table rows for primary metals
iso_rows = []
for _, row in iso_corr[iso_corr['metal'].isin(PRIMARY_METALS_FDR)].iterrows():
    rho = f"{row['rho']:.4f}" if pd.notna(row['rho']) else 'NA'
    p = f"{row['p_raw']:.4g}" if pd.notna(row['p_raw']) else 'NA'
    q = f"{row.get('p_fdr', np.nan):.4g}" if pd.notna(row.get('p_fdr', np.nan)) else 'NA'
    dir_ok = 'Yes' if pd.notna(row['rho']) and row['rho'] > 0 else ('No' if pd.notna(row['rho']) else 'NA')
    iso_rows.append(f"| {row['metal']} | isolate | {int(row['n'])} | {rho} | {p} | {q} | {dir_ok} |")

for _, row in well_corr[well_corr['metal'].isin(PRIMARY_METALS_FDR)].iterrows():
    rho = f"{row['rho']:.4f}" if pd.notna(row['rho']) else 'NA'
    p = f"{row['p_raw']:.4g}" if pd.notna(row['p_raw']) else 'NA'
    q = f"{row.get('p_fdr', np.nan):.4g}" if pd.notna(row.get('p_fdr', np.nan)) else 'NA'
    dir_ok = 'Yes' if pd.notna(row['rho']) and row['rho'] > 0 else ('No' if pd.notna(row['rho']) else 'NA')
    iso_rows.append(f"| {row['metal']} | well (≥2) | {int(row['n'])} | {rho} | {p} | {q} | {dir_ok} |")

table_content = chr(10).join(iso_rows)

enigma_section = f'''
---

## Exploratory Site-Level Validation — ENIGMA Isolates (Notebook 13)

**Analysis:** Spearman ρ (KO density vs well metal concentration) at isolate and well level.  
**Gene list:** Tier 1+2 (140 KOs), per-Mb density from isolate genomes (not MAGs).  
**Dataset:** `enigma_genome_depot_enigma.browser_genome` (strain_id IS NOT NULL = isolates) ×  
`enigma_coral.ddt_brick0000007` (groundwater metals, well-level median).  
**Pre-specified direction:** ρ > 0 (positive — isolates from high-metal wells carry more metal-gene KOs).  
**Note:** Exploratory. Results cannot bear on H1 (niche breadth). Null/opposite reported honestly.  
n_isolates = {n_iso}, n_wells = {n_wells}.

| Metal | Level | n | ρ | p_raw | p_FDR (primary) | Dir consistent |
|-------|-------|---|---|-------|-----------------|----------------|
{table_content}

**Interpretation:** See notebook NB13 for full results including secondary metals (As, Mn, Cr, Co).  
Well-level analysis limited to wells with ≥2 isolates; small n makes inference unreliable.
'''

if 'Exploratory Site-Level Validation — ENIGMA Isolates' not in content:
    with open(interp_path, 'a') as f:
        f.write(enigma_section)
    print(f'Appended ENIGMA isolate section to {interp_path}')
else:
    print('ENIGMA isolate section already present — not overwriting.')